<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
<br>汉化的库: <a href="https://github.com/GoatCsu/CN-LLMs-from-scratch.git">https://github.com/GoatCsu/CN-LLMs-from-scratch.git</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第五章: 在无标签数据集上预训练

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # For OpenAI's pretrained weights
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")
#同样导入库并检查版本

- 在本章中，我们将实现循环训练和基本模型评价的代码，用于预训练大语言模型。
- 在本章的最后，我们还将从 OpenAI 加载公开可用的预训练权重到我们的模型中。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/chapter-overview.webp" width=500px>

- 本章节涉及的主题如下所示

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model--0.webp" width=400px>

## 5.1 评估文本生成大模型

- 本节开始时，我们简要回顾了如何使用上一章的代码初始化 GPT 模型。
- 然后，我们讨论了大语言模型的基本评估指标。
- 最后，在本节中，我们将这些评估指标应用于训练和验证数据集。

### 5.1.1 用GPT来生成文本

- 我们首先与前几章一样初始化GPT

In [ ]:
import torch
from previous_chapters import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Disable dropout during inference
#导入模型, 设定一系列参数, 设定随机种子确保可复现

- 我们在上述代码中使用了 0.1 的 dropout 率，但如今训练大语言模型时通常不使用 dropout。
- 现代的大语言模型不在 `nn.Linear` 层的查询、键和值矩阵中使用偏置向量（与早期的 GPT 模型不同），而是通过设置 `"qkv_bias": False` 实现。
- 我们将上下文长度（`context_length`）减少到仅 256 个 token，以减少训练模型时的计算资源需求，而原始的 1.24 亿参数的 GPT-2 模型使用了 1024 个token。
  - 这是为了让更多读者可以在他们的笔记本电脑上运行并跟随代码示例。
  - 然而，您可以自由将 `context_length` 增加到 1024 个 token（这不需要更改任何代码）。
  - 我们稍后也将从预训练权重中加载一个具有 1024 `context_length` 的模型。

- 接下来，我们使用上一章中的 `generate_text_simple` 函数生成文本。
- 此外，我们定义了两个便利函数，`text_to_token_ids` 和 `token_ids_to_text`，用于在 token ID 和文本表示之间进行转换，这两个函数将在本章中多次使用。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-process.webp" width=500px>

In [ ]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor
#给输入的字符进行编码并实现一个Batch维度的向量,符合模型的输入形式
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())
#反向编码,去掉移除张量中的批次维度, 变成普通的链表
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
#举个例子
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    #初始上下文的Token ID张量，是上一步 text_to_token_ids 的输出
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)
#输出最长单词度为10的句子
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- 如上所示，由于模型尚未训练，它生成的文本并不理想。
- 我们如何衡量或设定“好文本”的标准，并将其转化为数值，以便在训练过程中进行跟踪？
- 下一小节介绍了计算生成输出的损失指标的度量标准，我们可以用它来衡量训练进度。
- 后续关于微调大语言模型的章节还将介绍其他衡量模型质量的方法。

### 5.1.2 计算文本生成的损失：交叉熵(cross- entropy)和困惑度(perplexity)

- 假设我们有一个 `inputs` 张量，其中包含 2 个训练示例（行）的 token ID。
- 与 `inputs` 对应，`targets` 包含我们希望模型生成的目标 token ID。
- 请注意，`targets` 是将 `inputs` 向右移动 1 个位置后的结果，正如我们在第二章实现数据加载器时所解释的那样。

In [ ]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]
#用向量的形式展现输入的文本
targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]
#用向量的形式展现要输出的东西

- 将 `inputs` 输入给模型后，我们将得到 2 个输入示例的 logits 向量，每个输入示例包含 3 个token。
- 每个 token 都是一个 50,257 维的向量，对应于词汇表的大小。
- 通过应用 softmax 函数，我们可以将 logits 张量转换为一个相同维度的张量，其中包含概率得分。

In [ ]:
with torch.no_grad():
    logits = model(inputs)
#不用梯度计算的计算inputes并储存
probas = torch.softmax(logits, dim=-1) # Probability of each token in vocabulary
#用soft Max整理logits
print(probas.shape) # Shape: (batch_size, num_tokens, vocab_size)

- 下图展示了我们如何将概率得分转换为文本，示例使用了一个非常小的词汇表，这一内容已在上一章的结尾讨论过。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-to-text.webp" width=500px>

- 如上一章所讨论的，我们可以应用 `argmax` 函数将概率得分转换为预测的 token ID。
- 上面的 softmax 函数为每个 token 生成了一个 50,257 维的向量；`argmax` 函数返回该向量中概率得分最高的位置，即给定 token 的预测 token ID。

- 由于我们有 2 个输入批次，每个批次包含 3 个 token，因此我们得到 2 行 3 列的预测 token ID：

In [ ]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
#相当于用贪心算法给出最有可能的答案
print("Token IDs:\n", token_ids)

- 如果解码这些 token，我们会发现它们与希望模型预测的 token (目标 token)有很大不同：

In [ ]:
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
#给出答案
print(f"Outputs batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")
#给出事实上的结论

- 这是因为模型还没有经过训练。
- 要训练模型，我们需要知道它与正确预测（目标）之间的差距有多大。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/proba-index.webp" width=500px>

- 对应于目标索引的 token 概率如下：

In [ ]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

- 我们希望最大化这些值，使它们的概率接近 1。
- 在数学优化中，最大化概率得分的对数比直接最大化概率得分更为简单；虽然这超出了本书的讨论范围，但我录制了一节更详细的讲座，您可以在这里查看：[L8.2 Logistic Regression Loss Function](https://www.youtube.com/watch?v=GxJe0DZvydM)。

In [ ]:
# Compute logarithm of all token probabilities
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)
#用对数输出他最大的可能数值

- 接下来，我们计算平均对数概率：

In [ ]:
# Calculate the average probability for each token
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)
#对数概率平均值

- 目标是通过优化模型权重，使得平均对数概率尽可能大。
- 由于对数的性质，最大的可能值是 0，而我们当前距离 0 还有很大差距。

- 在深度学习中，通常的做法是最小化 "负" 的平均对数概率值，而不是最大化平均对数概率值；在我们的例子中，深度学习中我们会最小化 10.7722 使其接近 0，而不是最大化 -10.7722 使其接近 0。
- 值 -10.7722 的负数，即 10.7722，在深度学习中也被称为交叉熵损失（cross-entropy loss）。

In [ ]:
neg_avg_log_probas = avg_log_probas * -1
#最大化对数等价为最小化负对数
print(neg_avg_log_probas)

- PyTorch 中的`cross_entropy` 已经能实现这些功能

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/cross-entropy.webp?123" width=400px>

- 在使用`cross_entropy` 之前, 我们可以看一下loggias跟target是怎样的

In [ ]:
# Logits have shape (batch_size, num_tokens, vocab_size)
print("Logits shape:", logits.shape)

# Targets have shape (batch_size, num_tokens)
print("Targets shape:", targets.shape)

- 有了PyTorch 中的 `cross_entropy` 函数，我们希望通过在批次维度上合并这些张量来将其展平：

In [ ]:
logits_flat = logits.flatten(0, 1)
#将张量 logits 的 第0维和第1维合并为一个维度，展平成一个二维张量
targets_flat = targets.flatten()
#将张量 targets 展平为一维张量

print("Flattened logits:", logits_flat.shape)
print("Flattened targets:", targets_flat.shape)

- 请注意，目标是 token ID，这些 ID 也代表我们希望最大化的 logits 张量中的索引位置。
- PyTorch 中的 `cross_entropy` 函数会自动处理对这些token索引的 softmax 和对数概率计算，确保它们被最大化。

In [ ]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)
#封装函数出马,一个代替好几行

- 与交叉熵损失相关的一个概念是大语言模型的困惑度 (perplexity)。
- 困惑度就是交叉熵损失的指数值。

In [ ]:
perplexity = torch.exp(loss)
#指数化loss作为P值
print(perplexity)

- 困惑度通常被认为更易解释，因为它可以理解为模型在每一步对词汇表大小的不确定性（在上面的例子中，这相当于 48,725 个单词或 token）。
- 换句话说，困惑度提供了一个衡量模型预测的概率分布与数据集中单词实际分布匹配程度的指标。
- 类似于损失值，较低的困惑度表示模型预测与实际分布的差距较小。

### 5.1.3 计算训练集和验证集的损失

- 我们使用一个相对较小的数据集来训练大语言模型（训练内容只有一个短篇故事）。
- 选择小故事的原因包括：
  - 无独立显卡的电脑也可以快速完成
  - 训练时间较短（以分钟计算，而非数周）
  - 我们使用了无版权文本，可以将其包含在这个 GitHub 仓库中，而不会违反任何使用权或显著增加仓库大小。

- 例如，Llama 2 7B 模型在 2 万亿 token 上训练时需要 184,320 个 GPU 小时（使用 A100 GPU）。
  - 截至本文撰写时，AWS 上 8xA100 云服务器的每小时成本约为 30 美元。
  - 因此，通过简单计算，训练这个 LLM 的成本为 184,320 / 8 * 30 美元 = 690,000 美元。

- 下面，我们使用了第二章中使用的同一数据集。

In [ ]:
import os
import urllib.request

file_path = "../../the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
#引入数据集
if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
#一系列经典的读取数据操作

- 通过前100个词与后100个词快速检查文本是否加载正常

In [ ]:
# First 100 characters
print(text_data[:99])

In [ ]:
# Last 100 characters
print(text_data[-99:])

In [ ]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
#统计一下文本的长度,编码文本内容并输出文本个数
print("Characters:", total_characters)
print("Tokens:", total_tokens)

- 为了教学,我们选取了这个短文本作为样例

- 接下来，我们将数据集划分为训练集和验证集，并使用第二章中的data loader为大语言模型（LLM）训练准备数据。
- 出于可视化的目的，下图假设 `max_length=6`，但对于训练加载器，我们将 `max_length` 设置为 LLM 支持的上下文长度。
- 为了简化，下图仅展示了输入token：
    - 由于我们训练 LLM 预测文本中的下一个单词，因此目标 token 与输入 token 相同，只是目标 token 向右移动了一个位置。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/batching.webp" width=500px>

In [ ]:
from previous_chapters import create_dataloader_v1
#从一个库导入之前的文章
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]
#这边可以手动定义训练集跟测试剂的比例

torch.manual_seed(123)
#依旧保持可复现
train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)
#初始化输入训练模型,给出批处理的大小、给出最大文本容量防止溢出
#给出不畅,丢弃最后一批不足的文本,打开随机防止拟合过度
val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)
#验证数据集仅仅修改了是否丢弃跟随抽取

In [ ]:
# Sanity check
# 神圣性,看一下一批次够了没

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

- 小的批处理数据集很适合用来小试牛刀
- 例如, Llama 27B就是用每次1024的批处理数据

- 另一种确认数据正常导入的方法如下

In [ ]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

- 还有一个方法来确认是否导入成功

In [ ]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch. numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()
#每次加一下训练数据集所有元素的种类
print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

# 在 PyTorch 中，调用 .numel() 方法会返回张量中所有元素的总数，无论张量的形状或维度如何

- 我们用了预分装函数来计算交叉熵
- 我们还调用另一个辅助函数，用于计算数据加载器中由用户指定的批次数Loss。

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    #呼唤GPU
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    #用交叉熵函数对于logits进行计算并且拉伸到二维长度
    return loss
#一个计算批损失的函数

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # 如果指定的批次数超过数据加载器中的总批次数，则将批次数减少到与数据加载器的总批次数匹配。
        num_batches = min(num_batches, len(data_loader))
        #减少需要处理的数量,同时也是防止溢出
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
        #一点点加上去的损失
    return total_loss / num_batches

- 如果你的电脑有支持 CUDA 的 GPU，大预言模型将无需更改代码即可在 GPU 上进行训练。
- 通过 `device` 设置，我们确保数据加载到与大语言模型相同的设备上。

In [ ]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
# 如果支持，则调用 GPU

# 注意：
# 如果取消注释以下代码块，代码可以在 Apple Silicon 芯片上运行（如果适用），
# 在 M3 MacBook Air 上测量速度大约是 Apple CPU 的两倍。
# 然而，计算得到的损失值可能会略有不同。

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")

model.to(device)  # 对于 nn.Module 类，不需要赋值 model = model.to(device)

torch.manual_seed(123)  # 固定随机种子，保证数据加载器打乱数据的结果可复现

with torch.no_grad():  # 禁用梯度跟踪以提高效率，因为此时尚未开始训练
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

# 推理阶段不计算梯度
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-1.webp" width=400px>

## 5.2 训练大语言模型

- 在本节中，我们最终实现了用于训练大语言模型（LLM）的代码。
- 我们想要于一个简单的训练函数（如果您对通过更高级的技术增强此训练函数感兴趣，例如学习率预热(rate warmup)、余弦退火(cosine annealing)和梯度裁剪(gradient clipping)，请参阅[附录D](../../appendix-D/01_main-chapter-code)）。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/train-steps.webp" width=300px>

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    #初始化训练模型而且给了空的队列
    # Main training loop
    for epoch in range(num_epochs):#训练次数
        model.train()  # Set model to training mode
        #转移到训练模块
        for input_batch, target_batch in train_loader:
            #从loader里面调出输入跟目标
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            #清空所有函数的梯度
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            #计算损失函数
            loss.backward() # Calculate loss gradients
            #反向传播优化
            optimizer.step() # Update model weights using loss gradients
            #更新权重
            tokens_seen += input_batch.numel()
            #加一下一共有多少
            global_step += 1
            #看一下一共训练了多少步
            # Optional evaluation step
            if global_step % eval_freq == 0:
                #按照一定的步数进行记录
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                #计算损失函数
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                #加到list中
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    #评价模块
    model.eval()
    #检验模式
    with torch.no_grad():
        #我认为的双保险,防止梯度更新
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    #	在评估结束后切换回训练模式，确保模型能继续用于训练。
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

(GPT的解释)
- global_step 是训练循环中的重要计数器，主要用于控制学习率调度、记录日志、保存模型检查点和控制终止条件等任务。
- 在分批训练中，它提供了一个统一的参考点，有助于管理复杂的训练流程。
- epoch 是按完整数据集的迭代单位，而 global_step 是按 batch 单位，粒度更细，用于管理更精确的任务。例如：
    1.	动态学习率调整,某些学习率调度器需要以 batch 为单位进行调整，而不是每个 epoch。例如，WarmUp 会在固定的前 N 步逐渐升高学习率。
	2.	频繁日志记录,记录训练日志时，通常是每隔 log_interval 步输出一次，而不是每个 epoch。
	3.	检查点保存,保存模型状态通常是按步数完成，尤其是当训练需要中断和恢复时，global_step 是更精确的 token。

- 我们用上述的定义训练一下这个模型

In [ ]:
# Note:
# Uncomment the following code to calculate the execution time
#下面可以看一下计算了多久
# import time
# start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
#经典操作
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
#用Adam进行优化,其中学习rate为0.004,动量衰减是0.1
num_epochs = 10
#10论学习
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=tokenizer
)
#记录了开始文本、检验的频率
# 注意：
# 如果需要显示执行时间，请取消注释以下代码
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"训练完成耗时 {execution_time_minutes:.2f} 分钟。")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    # Plot training and validation loss against epochs
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))  # only show integer labels on x-axis

    # Create a second x-axis for tokens seen
    ax2 = ax1.twiny()  # Create a second x-axis that shares the same y-axis
    ax2.plot(tokens_seen, train_losses, alpha=0)  # Invisible plot for aligning ticks
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()  # Adjust layout to make room
    plt.savefig("loss-plot.pdf")
    plt.show()

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
#一个经典的plot画图函数

- 从以上结果可以看出，模型在开始阶段生成的是难以理解的字符串，但是在后期能够生成基本符合语法的句子。
- 从训练集和验证集的损失值可以看出，模型开始出现过拟合现象。
- 如果检查后期它生成的某些段落，会发现它们与训练集中的内容完全相同(模型只是简单地记住了训练数据,背住答案罢了)。
- 之后的部分，我们将讨论一些解码策略，这些策略可以一定程度缓解这种“背答案”的问题。
- 请注意，这里的过拟合是由于训练集非常非常小，并且我们对其进行了多次迭代。
  - 本次 LLM 训练主要用于教学目的；我们的目标是观察模型是否能够学会生成连贯的文本。
  - 为了避免花费数周或数月时间在大量昂贵硬件上训练模型，我们将在后续加载预训练权重。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-2.webp" width=350px>

**如果您对通过更高深的技术增强此训练函数感兴趣，例如学习率预热、余弦退火和梯度裁剪，请参阅[附录D](../../appendix-D/01_main-chapter-code)。**

**更大的数据集跟更深度的训练,可以在以下找到链接 [../03_bonus_pretraining_on_gutenberg](../03_bonus_pretraining_on_gutenberg)**

## 5.3 控制随机性的解码策略

- 对于像我们训练的这种规模相对较小的GPT模型（LLM），推理阶段的计算成本较低。因此，如果在训练时使用了GPU，推理阶段则无需依赖GPU资源。
- 我们可以利用第5章介绍的`generate_text_simple`函数（该函数已在简单训练函数中被调用）来逐步生成新文本，每次生成一个单词（或 token）。
- 正如5.1.2节所提到的，下一个生成的 token 是从词汇表中选取概率得分最高的 token 。

In [ ]:
model.to("cpu")
model.eval()

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=25,
    context_size=GPT_CONFIG_124M["context_length"]
)
#经典的载入
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- 即使我们多次调用 `generate_text_simple` 函数，大语言模型（LLM）生成的输出也始终是确定性的，即每次结果相同。
- 为了增强生成文本的灵活性，我们引入了两种解码策略来改进 `generate_text_simple`：**温度缩放** 和 **top-k 采样**。
- 这些方法能够调节模型生成文本的随机性和多样性，从而满足不同的应用需求。

### 5.3.1 温度缩放

- 在之前的实现中，我们始终使用 `torch.argmax` 来选择概率最高的 token 作为下一个生成的 token。
- 为了增加生成文本的多样性，我们可以改用 `torch.multinomial(probs, num_samples=1)`，从概率分布中随机采样下一个 token。
- 在这种方法中，每个索引被选中的概率与其在输入张量中对应的概率值成正比，从而实现基于概率的随机采样。

- 以下是对生成下一个 token 过程的简单回顾，假设我们使用一个非常小的词汇表来说明：

In [ ]:
vocab = { 
    "closer": 0,
    "every": 1, 
    "effort": 2, 
    "forward": 3,
    "inches": 4,
    "moves": 5, 
    "pizza": 6,
    "toward": 7,
    "you": 8,
} 

inverse_vocab = {v: k for k, v in vocab.items()}
#插入
# Suppose input is "every effort moves you", and the LLM
# returns the following logits for the next token:
next_token_logits = torch.tensor(
    [4.51, 0.89, -1.90, 6.75, 1.63, -1.62, -1.89, 6.28, 1.79]
)

probas = torch.softmax(next_token_logits, dim=0)
#softmax归一化
next_token_id = torch.argmax(probas).item()
#选个可能性最大
# The next generated token is then as follows:
print(inverse_vocab[next_token_id])

In [ ]:
torch.manual_seed(123)
next_token_id = torch.multinomial(probas, num_samples=1).item()
print(inverse_vocab[next_token_id])

- 我们不再依赖 `torch.argmax` 来选择最可能的 token ，而是通过 `torch.multinomial(probas, num_samples=1)` 从 softmax 分布中采样来确定下一个 token 。
- 为了直观地理解这一过程，我们可以使用原始的 softmax 概率对下一个 token 进行 1,000 次采样，并观察结果分布：

In [ ]:
def print_sampled_tokens(probas):
    torch.manual_seed(123) # Manual seed for reproducibility
    sample = [torch.multinomial(probas, num_samples=1).item() for i in range(1_000)]
    #从概率分布 probas 中按照权重进行一次采样,并生成索引
    sampled_ids = torch.bincount(torch.tensor(sample))
    #然后变成单词
    for i, freq in enumerate(sampled_ids):
        print(f"{freq} x {inverse_vocab[i]}")
#统计采样过程中每个词的出现频率
print_sampled_tokens(probas)

- 我们可以通过一种称为**温度缩放**的技术来调节概率分布和 token 选择的过程。
- 温度缩放的核心操作是将 logits 除以一个大于 0 的数值（即温度值），然后再应用 softmax 函数。
- 当温度值大于 1 时，softmax 输出的概率分布会更加均匀，从而增加生成文本的多样性。
- 当温度值小于 1 时，softmax 输出的概率分布会更加集中（更陡峭或更尖锐），从而倾向于选择概率更高的 token，减少随机性。

模型的预测概率往往过于自信或低估某些类别的概率，尤其在分类任务中。
温度缩放通过引入一个参数  T > 0  来重新调整 logits，改善预测概率的校准性能

In [ ]:
def softmax_with_temperature(logits, temperature):
    scaled_logits = logits / temperature
    return torch.softmax(scaled_logits, dim=0)
#温度校正
# Temperature values
temperatures = [1, 0.1, 5]  # Original, higher confidence, and lower confidence
#初始校正系数
# Calculate scaled probabilities
scaled_probas = [softmax_with_temperature(next_token_logits, T) for T in temperatures]

In [ ]:
# Plotting
x = torch.arange(len(vocab))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(5, 3))
for i, T in enumerate(temperatures):
    rects = ax.bar(x + i * bar_width, scaled_probas[i], bar_width, label=f'Temperature = {T}')

ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(vocab.keys(), rotation=90)
ax.legend()

plt.tight_layout()
plt.savefig("temperature-plot.pdf")
plt.show()
#一套经典的画图

- 从结果中可以看出，当温度设置为 0.1 时，概率分布变得更加陡峭，接近于 `torch.argmax` 的行为，因此最可能的 token 几乎总是被选中：

In [ ]:
print_sampled_tokens(scaled_probas[1])

- 当温度设置为 5 时，概率分布变得更加均匀，从而增加了生成文本的多样性和随机性：

In [ ]:
print_sampled_tokens(scaled_probas[2])

- 假设大语言模型（LLM）的输入是“every effort moves you”，上述方法有时可能会生成无意义的文本，例如“every effort moves you pizza”，其出现的概率为 3.2%（即在 1000 次采样中出现了 32 次）。

### 5.3.2 Top-k 取样

- 为了在使用更高温度增加输出多样性的同时减少生成无意义句子的概率，我们可以将采样限制在前 k 个最可能的 token 中：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/topk.webp" width=500px>

- （请注意，此图中的数值已截取到小数点后两位，以减少视觉干扰。Softmax 行中的值总和应为 1.0。）

- 我们可以按照下述建议补充代码

-	控制输出质量： 减少低概率、无意义的词被选中的机会。
-	保持多样性： 允许模型在概率较高的几个候选词中随机选择，而不是总是选择最高概率的词（这会导致输出缺乏变化）。

In [ ]:
top_k = 3
top_logits, top_pos = torch.topk(next_token_logits, top_k)
#topK采样
print("Top logits:", top_logits)
print("Top positions:", top_pos)

In [ ]:
new_logits = torch.where(
    condition=next_token_logits < top_logits[-1],
    input=torch.tensor(float("-inf")), 
    other=next_token_logits
)
#不是前K遮蔽掉
print(new_logits)

> NOTE:  
>
>  一种稍微更高效的实现方式可以通过以下代码实现：
>
> ```python
> new_logits = torch.full_like( # create tensor containing -inf values
>    next_token_logits, -torch.inf
>)   
> new_logits[top_pos] = next_token_logits[top_pos] # copy top k values into the -inf tensor
> ```
>
> For more details, see https://github.com/rasbt/LLMs-from-scratch/discussions/326


In [ ]:
topk_probas = torch.softmax(new_logits, dim=0)
print(topk_probas)

### 5.3.3 优化文本更新功能

- 在前两小节中，我们介绍了**温度采样**和**top-k 采样**的概念。
- 现在，我们将结合这两种方法，对之前用于生成大语言模型（LLM）文本的 `generate_simple` 函数进行改进，创建一个新的 `generate` 函数：

    - (译者):用自己的话总结下
    - 温度校正是更加平滑,防止数据差之毫厘以谬以千里 
    - topK是防止臭鱼烂虾进入筛选范围提高质量

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
#生成模块
    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        #计算预测值,但是切最后一个
        # New: Filter logits with top_k sampling
        #top K采样
        if top_k is not None:
            # Keep only top_k values
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)
            
        # New: Apply temperature scaling
        #温度校正
        if temperature > 0.0:
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = torch.softmax(logits, dim=-1)  # (batch_size, context_len)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (batch_size, 1)
            #从概率分布中采样下一个 token 

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch_size, 1)
            #如果未启用采样，选择概率最高的 token 作为下一个 token 
        if idx_next == eos_id:  # Stop generating early if end-of-sequence token is encountered and eos_id is specified
            break

        # Same as before: append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch_size, num_tokens+1)

    return idx

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", tokenizer),
    max_new_tokens=15,
    context_size=GPT_CONFIG_124M["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))
#经典的操作

## 5.4 在Pytorch中加载并保留权重

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/mental-model-3.webp" width=400px>

- 大模型的训练是很贵的, 所以导入已训练好的参数是很有必要的   
- 在Pytorch中我们所推荐的保存方式是所谓的 `state_dict` ,这玩意通过调用 `torch.save` 的子模块 `.state_dict()` :

In [ ]:
torch.save(model.state_dict(), "model.pth")
#训练完的数据保存一下

- 之后我们可以对新的 `GPTModel` 导入已经训练好的参数:

In [ ]:
model = GPTModel(GPT_CONFIG_124M)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
model.eval()

- 自适应的Adam跟AdamW相较于SGD更好!
- 但是这些算法需要另外的参数, 所以保存训练好的参数就更有必要了:

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    }, 
    "model_and_optimizer.pth"
)
#全家整整齐齐地保存

In [ ]:
checkpoint = torch.load("model_and_optimizer.pth", weights_only=True)
#保存检查点
model = GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train()
#调整到训练模式

## 5.5 从OpenAI导入超参数

- 在之前的实验中，我们仅使用了一本非常短的小故事书来训练一个小型 GPT-2 模型，这主要是为了教学目的。
- 对此感兴趣的读者可以在[../03_bonus_pretraining_on_gutenberg](../03_bonus_pretraining_on_gutenberg)中找到基于完整古登堡计划图书语料库的更长时间预训练记录。
- 幸运的是，我们无需花费数万到数十万美元在大型预训练语料库上预训练模型，而是可以直接加载 OpenAI 提供的预训练权重。

- 有关从Hugging Face中加载权重的另一种方法请参阅 [../02_alternative_weight_loading](../02_alternative_weight_loading)

- 首先，我们需要一些基础代码来从 OpenAI 下载文件并将权重加载到 Python 中。
- 由于 OpenAI 使用了 [TensorFlow](https://www.tensorflow.org/)，我们需要安装并使用 TensorFlow 来加载权重；同时，[tqdm](https://github.com/tqdm/tqdm) 是一个用于显示进度条的库。
- 取消注释并运行下一个代码单元以安装所需的库。

In [ ]:
# pip install tensorflow tqdm

In [ ]:
print("TensorFlow version:", version("tensorflow"))
print("tqdm version:", version("tqdm"))
#tensorflow他到底还是来了

In [ ]:
# Relative import from the gpt_download.py contained in this folder
from gpt_download import download_and_load_gpt2
#召唤神仙

- 通过如下代码下载124M的模型:

In [ ]:
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

In [ ]:
print("Settings:", settings)

In [ ]:
print("Parameter dictionary keys:", params.keys())

In [ ]:
print(params["wte"])
print("Token embedding weight tensor dimensions:", params["wte"].shape)

- 此外，`model_size` 参数还支持 "355M"、"774M" 和 "1558M" 等选项。
- 下图总结了这些不同规模模型之间的主要差异：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch05_compressed/gpt-sizes.webp?timestamp=123" width=500px>

- 在上述操作中，我们已经成功将 124M 的 GPT-2 模型权重加载到 Python 中，但仍需将这些权重传输到我们的 `GPTModel` 实例中。
- 首先，我们需要初始化一个新的 `GPTModel` 实例。
- 需要注意的是，原始的 GPT 模型在多头注意力模块中为查询、键和值矩阵的线性层初始化了带有偏置向量的权重，这种做法既不必要也不推荐。然而，为了正确加载权重，我们在实现中必须将 `qkv_bias` 参数设置为 `True`。
- 此外，我们使用了原始 GPT-2 模型所支持的 `1024`  token 的上下文长度。

In [ ]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
#把每个大小的模型都与先载入并确定好
# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval()

- 接下来的任务是将 OpenAI 的权重分配到我们 `GPTModel` 实例中对应的权重张量中。

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    #位置权重
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    #单词全中
    for b in range(len(params["blocks"])):
        #三个参数的输入
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight, 
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias, 
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight, 
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias, 
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight, 
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias, 
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, 
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, 
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, 
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, 
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])
    
#主要目的是将预训练的模型参数加载到一个gpt中
load_weights_into_gpt(gpt, params)
gpt.to(device);

- 如果模型正确加载了,我们可以用先前的`generate` :

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=gpt,
    idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
    max_new_tokens=25,
    context_size=NEW_CONFIG["context_length"],
    top_k=50,
    temperature=1.5
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

- 我们可以确认模型权重已正确加载，因为模型能够生成连贯的文本；如果我们在加载过程中出现任何错误，模型将无法实现这一点。

- 如果您想了解另一种从 Hugging Face Hub 加载权重的方法，请参阅 [../02_alternative_weight_loading](../02_alternative_weight_loading)。

- 如果您对 GPT 架构与 Llama 架构（Meta AI 开发的一种流行大语言模型）之间的比较感兴趣，请查看附加内容：[../07_gpt_to_llama](../07_gpt_to_llama)。

## 总结与收获

- 请参考 [./gpt_train.py](./gpt_train.py) 脚本，这是一个独立的训练脚本。
- [./gpt_generate.py](./gpt_generate.py) 脚本会加载 OpenAI 提供的预训练权重，并根据提示生成文本。
- 您可以在 [./exercise-solutions.ipynb](./exercise-solutions.ipynb) 中找到练习的解答。